# Tesla and GameStop Stock and Revenue Data Analysis Using Python

**Project:** IBM Coursera – Python Project for Data Science

This notebook retrieves stock-price data, extracts revenue tables, and visualizes stock-price and revenue trends for Tesla and GameStop.

> Run the notebook from top to bottom. Internet access is required for data downloads.

## Setup

Install the required packages if they are not already available in your environment.

In [ ]:
%pip -q install yfinance pandas requests beautifulsoup4 lxml plotly

## Question 1 – Extract Tesla Stock Data

In [ ]:
import yfinance as yf
import pandas as pd

tesla_data = yf.download("TSLA", period="max", auto_adjust=False, progress=False)

# Flatten columns if yfinance returns a multi-level column index
if isinstance(tesla_data.columns, pd.MultiIndex):
    tesla_data.columns = tesla_data.columns.get_level_values(0)

tesla_data.reset_index(inplace=True)
tesla_data.to_csv("tesla_stock_data.csv", index=False)

tesla_data.head()

## Question 2 – Extract Tesla Revenue Data

In [ ]:
import requests
from bs4 import BeautifulSoup

def get_revenue_table(url, company_name):
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()

    tables = pd.read_html(response.text)
    revenue_table = None

    for table in tables:
        columns = [str(c).strip().lower() for c in table.columns]
        if any("revenue" in c for c in columns) or (
            len(table.columns) >= 2 and "date" in " ".join(columns)
        ):
            revenue_table = table.copy()
            break

    if revenue_table is None and tables:
        revenue_table = tables[0].copy()

    if revenue_table is None:
        raise ValueError(f"No table was found for {company_name}. Check whether the webpage is accessible.")

    revenue_table.columns = [str(c).strip() for c in revenue_table.columns]
    return revenue_table.dropna(how="all")

tesla_revenue = get_revenue_table(
    "https://www.macrotrends.net/stocks/charts/TSLA/tesla/revenue",
    "Tesla"
)
tesla_revenue.tail()

## Question 3 – Extract GameStop Stock Data

In [ ]:
gme_data = yf.download("GME", period="max", auto_adjust=False, progress=False)

if isinstance(gme_data.columns, pd.MultiIndex):
    gme_data.columns = gme_data.columns.get_level_values(0)

gme_data.reset_index(inplace=True)
gme_data.to_csv("gamestop_stock_data.csv", index=False)

gme_data.head()

## Question 4 – Extract GameStop Revenue Data

In [ ]:
gme_revenue = get_revenue_table(
    "https://www.macrotrends.net/stocks/charts/GME/gamestop/revenue",
    "GameStop"
)
gme_revenue.tail()

## Shared graph function for Questions 5 and 6

The function below plots historical closing share price and revenue. It expects the revenue table to contain a date column and a revenue column. If the source website changes its table format, inspect `tesla_revenue.columns` or `gme_revenue.columns` and adjust the column selection.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def make_graph(stock_data, revenue_data, company_name):
    stock = stock_data.copy()
    revenue = revenue_data.copy()

    # Identify date and revenue columns
    date_col = next((c for c in revenue.columns if "date" in str(c).lower()), revenue.columns[0])
    revenue_col = next((c for c in revenue.columns if "revenue" in str(c).lower()), revenue.columns[-1])

    stock["Date"] = pd.to_datetime(stock["Date"], errors="coerce")
    revenue["Date"] = pd.to_datetime(revenue[date_col], errors="coerce")
    revenue["Revenue"] = (
        revenue[revenue_col].astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.replace("--", "", regex=False)
    )
    revenue["Revenue"] = pd.to_numeric(revenue["Revenue"], errors="coerce")

    stock = stock[stock["Date"] <= "2021-06-30"]
    revenue = revenue[revenue["Date"] <= "2021-06-30"]

    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True,
        subplot_titles=("Historical Share Price", "Historical Revenue"),
        vertical_spacing=0.3
    )
    fig.add_trace(
        go.Scatter(x=stock["Date"], y=pd.to_numeric(stock["Close"], errors="coerce"),
                   name="Share Price"),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(x=revenue["Date"], y=revenue["Revenue"], name="Revenue"),
        row=2, col=1
    )
    fig.update_xaxes(title_text="Date", row=2, col=1)
    fig.update_yaxes(title_text="Share Price (US$)", row=1, col=1)
    fig.update_yaxes(title_text="Revenue (US$ millions)", row=2, col=1)
    fig.update_layout(title=f"{company_name} Stock and Revenue Dashboard",
                      showlegend=False, height=800)
    fig.show()

## Question 5 – Plot Tesla Stock Graph

In [ ]:
make_graph(tesla_data, tesla_revenue, "Tesla")

## Question 6 – Plot GameStop Stock Graph

In [ ]:
make_graph(gme_data, gme_revenue, "GameStop")

## Submission

1. Run all cells and confirm that the tables and graphs display.
2. Save this notebook as `Tesla_GameStop_Stock_Revenue_Analysis.ipynb`.
3. Upload the notebook to a GitHub repository.
4. Open the uploaded `.ipynb` file on GitHub and copy its browser URL.
5. Paste that URL into the Coursera submission field.

**Note:** The notebook file itself is not a GitHub link. You must upload it to your own GitHub account first, then copy the resulting URL.